# Phase 3: NASA C-MAPSS 센서 데이터 기반 터보팬 엔진 예지보전 (Predictive Maintenance)

이 노트북은 NASA C-MAPSS (Turbofan Engine Degradation Simulation) 데이터셋을 활용하여 LSTM(Long Short-Term Memory) 기반의 RUL(Remaining Useful Life) 예측 모델을 구축하는 전체 과정을 담고 있습니다.

**핵심 파이프라인:**
1. 데이터 로드 및 전처리 (라벨링)
2. Min-Max 스케일링
3. 시계열 슬라이딩 윈도우 (Sliding Window) 텐서 변환
4. LSTM 딥러닝 모델 아키텍처 설계 및 훈련
5. 예측 성능(RMSE) 평가 및 시각화

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout

import warnings
warnings.filterwarnings('ignore')

### 1. 데이터 로드 및 RUL 라벨링
C-MAPSS의 FD001 데이터셋을 다운로드하고 컬럼명을 지정합니다.

In [ ]:
# 다운로드 링크 (FD001)
train_url = 'https://raw.githubusercontent.com/CHADALAI/NASA-CMAPSS-Dataset/master/train_FD001.txt'
test_url = 'https://raw.githubusercontent.com/CHADALAI/NASA-CMAPSS-Dataset/master/test_FD001.txt'
rul_url = 'https://raw.githubusercontent.com/CHADALAI/NASA-CMAPSS-Dataset/master/RUL_FD001.txt'

# 컬럼명 설정
columns = ['id', 'cycle', 'setting1', 'setting2', 'setting3', 's1', 's2', 's3',
           's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14',
           's15', 's16', 's17', 's18', 's19', 's20', 's21']

train = pd.read_csv(train_url, sep=' ', header=None).drop([26,27], axis=1)
train.columns = columns

test = pd.read_csv(test_url, sep=' ', header=None).drop([26,27], axis=1)
test.columns = columns

y_test = pd.read_csv(rul_url, sep=' ', header=None).drop([1], axis=1)
y_test.columns = ['RUL']

print(f"Train shape: {train.shape}, Test shape: {test.shape}")

In [ ]:
# RUL (Remaining Useful Life) 계산
rul = pd.DataFrame(train.groupby('id')['cycle'].max()).reset_index()
rul.columns = ['id', 'max']
train = train.merge(rul, on=['id'], how='left')
train['RUL'] = train['max'] - train['cycle']
train.drop('max', axis=1, inplace=True)
train.head()

### 2. 시계열 전처리 및 슬라이딩 윈도우
LSTM에 입력하기 위해 `(Samples, Time Steps, Features)` 3D 형태로 변환합니다.

In [ ]:
# 센서 컬럼만 추출 (고정된 세팅값이나 의미 없는 센서 제외 가능, 여기서는 편의상 주요 센서 사용)
sensor_cols = ['s2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']
sequence_cols = ['setting1', 'setting2'] + sensor_cols

scaler = MinMaxScaler()
train[sequence_cols] = scaler.fit_transform(train[sequence_cols])
test[sequence_cols] = scaler.transform(test[sequence_cols])

def gen_sequence(id_df, seq_length, seq_cols):
    data_matrix = id_df[seq_cols].values
    num_elements = data_matrix.shape[0]
    for start, stop in zip(range(0, num_elements-seq_length), range(seq_length, num_elements)):
        yield data_matrix[start:stop, :]

def gen_labels(id_df, seq_length, label):
    data_matrix = id_df[label].values
    num_elements = data_matrix.shape[0]
    return data_matrix[seq_length:num_elements, :]

seq_length = 50

# 생성된 시퀀스 리스트
seq_gen = (list(gen_sequence(train[train['id']==id], seq_length, sequence_cols)) 
           for id in train['id'].unique())
seq_array = np.concatenate(list(seq_gen)).astype(np.float32)
print(f"X_train shape: {seq_array.shape}")

In [ ]:
label_gen = [gen_labels(train[train['id']==id], seq_length, ['RUL']) 
             for id in train['id'].unique()]
label_array = np.concatenate(label_gen).astype(np.float32)
print(f"y_train shape: {label_array.shape}")

### 3. LSTM 모델 학습

In [ ]:
nb_features = seq_array.shape[2]
nb_out = label_array.shape[1]

model = Sequential()
model.add(LSTM(input_shape=(seq_length, nb_features), units=100, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(units=50, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(units=nb_out))
model.compile(loss='mean_squared_error', optimizer='adam')

model.summary()

history = model.fit(seq_array, label_array, epochs=10, batch_size=200, validation_split=0.05, verbose=1)

### 4. 평가 및 시각화

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Training History')
plt.legend()
plt.show()